In [30]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
import os
from langchain_core.tools import tool
 
load_dotenv(override=True)
 
#print(os.environ.get("ANTHROPIC_API_KEY")[:3])  # Check if it's set
 
# Initialize the Claude model
llm = ChatAnthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL"),             # Required: model ID to use
    temperature=1.0,                       # Randomness: 0.0 = deterministic, 1.0 = default (max varies by model)
    max_tokens=4096,                       # Max tokens in response (default from model profile, fallback 4096)
    top_k=None,                            # Top-K sampling: consider only top K tokens (None = disabled)
    top_p=None,                            # Top-P sampling: nucleus sampling threshold (None = disabled)
    timeout=None,                          # Request timeout in seconds (None = no timeout)
    max_retries=2,                         # Number of retries on failed requests
    stop=None,                             # Stop sequences e.g. ["\n", "END"] (None = disabled)
    #base_url="https://api.anthropic.com",  # API base URL (can also set via ANTHROPIC_API_URL env var)
    streaming=True,                       # Stream tokens as generated (False = wait for full response)
    thinking=None,
    base_url=os.environ.get("ENDPOINT")                                          # Extended thinking e.g. {"type": "enabled", "budget_tokens": 5000}
)
print(llm.invoke("What is Agentic AI?"))
print("Set up is complete")

content='# Agentic AI\n\nAgentic AI refers to AI systems that can **autonomously pursue goals** with minimal human intervention. Here are the key characteristics:\n\n## Core Features\n\n- **Autonomous goal-setting & planning** - Decides what steps to take to achieve objectives\n- **Tool use** - Can access and use external tools, APIs, databases, or software\n- **Reasoning & iteration** - Works through multi-step problems, adjusts approach based on feedback\n- **Minimal supervision** - Operates independently rather than just responding to prompts\n\n## How It Differs From Current AI\n\n| Current AI (Assistive) | Agentic AI |\n|---|---|\n| Responds to requests | Pursues goals independently |\n| Single-turn interactions | Multi-step problem-solving |\n| Human-directed | Self-directed (within parameters) |\n\n## Real-World Examples\n\n- An AI that schedules your calendar by contacting attendees, checking availability, and booking rooms\n- A research agent that autonomously gathers data, ru

In [19]:
def genai_support_bot(question: str) -> str:
    """Simple GenAI bot - no memory, no tools, just Q&A"""
    messages = [
        SystemMessage(content="You are a helpful customer service assistant for ShopEasy e-commerce."),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

# Test the GenAI bot
print("=" * 60)
print("GenAI Bot Response:")
print("=" * 60)
print(genai_support_bot("What is the Cybersecurity?"))

GenAI Bot Response:
# Cybersecurity

Cybersecurity refers to the practice of protecting computer systems, networks, and data from digital attacks, theft, and unauthorized access. Here's an overview:

## Key Components

- **Network Security** - Protecting data in transit across networks
- **Application Security** - Securing software from vulnerabilities
- **Data Security** - Protecting sensitive information from breaches
- **Identity & Access Management** - Controlling who can access systems
- **Incident Response** - Reacting to and recovering from attacks

## Common Threats

- Malware and viruses
- Phishing attacks
- Ransomware
- Password breaches
- Unauthorized access

## Best Practices

- Use strong, unique passwords
- Enable multi-factor authentication (MFA)
- Keep software updated
- Regular backups
- Employee security training
- Firewalls and antivirus software

## At ShopEasy

We take cybersecurity seriously to protect your:
- Personal information
- Payment details
- Order history

In [20]:
print("Turn 1:")
print(genai_support_bot("My name is Sarah and my order number is #12345"))

print("\n" + "=" * 60)
print("Turn 2:")
print(genai_support_bot("What is my name and order number?"))

Turn 1:
Hello Sarah! 👋

Thank you for contacting ShopEasy! I have your order number **#12345** on file.

How can I help you today? I'm here to assist with:
- Order status and tracking
- Returns or exchanges
- Product questions
- Issues with your delivery
- Account or payment concerns
- Any other questions about your order

What's going on with your order?

Turn 2:
I don't have access to your personal information, account details, or order history. I'm a general customer service assistant without access to ShopEasy's customer database.

To help you, I would need you to provide:
- Your name
- Your order number

Alternatively, you can:
- Log into your ShopEasy account to view your order details
- Contact ShopEasy's customer service directly with your account information for verification
- Check your email confirmation for your order number

How can I assist you once you've provided these details?


In [23]:
# Simulated database (in real-world, this would be your actual database)
ORDERS_DB = {
    "12345": {
        "customer": "Sarah Johnson",
        "status": "Shipped",
        "items": ["Wireless Headphones", "Phone Case"],
        "tracking": "TRK-789456123",
        "delivery_date": "June 26, 2026"
    },
    "67890": {
        "customer": "Mike Chen",
        "status": "Processing",
        "items": ["Laptop Stand", "USB-C Hub"],
        "tracking": None,
        "delivery_date": "Pending"
    }
}

INVENTORY_DB = {
    "Wireless Headphones": {"stock": 45, "price": 79.99},
    "Phone Case": {"stock": 120, "price": 19.99},
    "Laptop Stand": {"stock": 0, "price": 49.99},  # Out of stock!
    "USB-C Hub": {"stock": 30, "price": 39.99}
}

print("Databases loaded!")

Databases loaded!


In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up order details by order ID. Returns order status, items, and tracking info."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        order = ORDERS_DB[order_id]
        return f"""Order #{order_id}:
- Customer: {order['customer']}
- Status: {order['status']}
- Items: {', '.join(order['items'])}
- Tracking: {order['tracking'] or 'Not yet available'}
- Expected Delivery: {order['delivery_date']}"""
    return f"Order #{order_id} not found in our system."

@tool
def check_inventory(product_name: str) -> str:
    """Check if a product is in stock and get its price."""
    for name, info in INVENTORY_DB.items():
        if product_name.lower() in name.lower():
            status = "In Stock" if info['stock'] > 0 else "Out of Stock"
            return f"{name}: {status} ({info['stock']} units) - ${info['price']}"
    return f"Product '{product_name}' not found."

@tool
def initiate_return(order_id: str, reason: str) -> str:
    """Initiate a return request for an order."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        return f"Return request initiated for Order #{order_id}. Reason: {reason}. Return label will be emailed within 24 hours."
    return f"Cannot initiate return - Order #{order_id} not found."

tools = [lookup_order, check_inventory, initiate_return]
print("Tools defined:", [t.name for t in tools])

Tools defined: ['lookup_order', 'check_inventory', 'initiate_return']


In [25]:
from langchain.agents import create_agent

# Create the Agentic AI bot with tools and memory

system_prompt = """You are a helpful customer service agent for ShopEasy e-commerce.
    
You have access to tools to:
- Look up order status and tracking
- Check product inventory
- Initiate returns

Always use the appropriate tool to get real information. Be friendly and helpful.
If you need to look something up, do it before responding."""

# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Memory store - now uses HumanMessage and AIMessage format
chat_history = []

def agentic_support_bot(user_input: str) -> str:
    """Agentic bot with tools and memory"""
    global chat_history
    
    # Build messages list: system + chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Agentic bot ready!")

Agentic bot ready!


In [26]:
# Test 1: Order lookup (Tool Use)
print("=" * 60)
print("Customer: What is the status of my order #12345?")
print("=" * 60)
response = agentic_support_bot("What is the status of my order #12345?")
print("\nAgent Response:")
print(response)

Customer: What is the status of my order #12345?

Agent Response:
[{'text': "Great! Here's the status of your order #12345:\n\n**Order Status:** Shipped ✓\n\n**Items in your order:**\n- Wireless Headphones\n- Phone Case\n\n**Tracking Information:** TRK-789456123\n\n**Expected Delivery Date:** June 26, 2026\n\nYour order is on its way! You can use the tracking number (TRK-789456123) to get more detailed delivery updates. Is there anything else I can help you with?", 'type': 'text', 'index': 0}]


In [27]:
print("=" * 60)
print("Customer: When will it arrive?")
print("=" * 60)
response = agentic_support_bot("When will it arrive?")
print("\nAgent Response:")
print(response)

Customer: When will it arrive?

Agent Response:
[{'text': 'Based on the order details, your order #12345 is expected to arrive on **June 26, 2026**.\n\nSince your order has already shipped with tracking number TRK-789456123, you should be able to get more specific delivery updates by tracking that number with the carrier. The tracking information may provide a more precise delivery window as the package gets closer to your location.\n\nIs there anything else I can help you with regarding your order?', 'type': 'text', 'index': 0}]


In [28]:
print("=" * 60)
print("Customer: I want to return the headphones. They don't fit well.")
print("=" * 60)
response = agentic_support_bot("I want to return the headphones. They don't fit well.")
print("\nAgent Response:")
print(response)

Customer: I want to return the headphones. They don't fit well.

Agent Response:
[{'text': "Perfect! I've successfully initiated a return request for the headphones in your order #12345. \n\n**Return Details:**\n- **Reason:** Headphones don't fit well\n- **Return Label:** You'll receive a return shipping label via email within 24 hours\n\nOnce you receive the return label, you can print it out and attach it to the package with the headphones. Just drop it off at your nearest shipping location, and we'll process your return once we receive it back.\n\nIs there anything else I can assist you with?", 'type': 'text', 'index': 0}]


In [29]:
response = agentic_support_bot("Is the Laptop Stand available?")
print(response)

DISCOUNT_CODES = {
    "SAVE10": {"discount": 10, "min_order": 50},
    "WELCOME20": {"discount": 20, "min_order": 100},
}

@tool
def validate_discount_code(code: str) -> str:
    """Validate a discount code and return the discount details."""
    # YOUR CODE HERE
    # 1. Check if code exists in DISCOUNT_CODES
    # 2. Return discount details or "Invalid code" message
    pass

[{'text': "Unfortunately, the **Laptop Stand** is currently **out of stock**. However, it's regularly priced at **$49.99** when it becomes available.\n\nI'd recommend:\n- Checking back soon, as we may restock it shortly\n- Signing up for a back-in-stock notification if that option is available on the product page\n\nWould you like me to help you with anything else, or would you like to look for a similar product?", 'type': 'text', 'index': 0}]
